In [1]:
# ============================================================
# STEP 0 — Verify actual native sampling rate of Empatica channels
# Run this FIRST, before touching architecture/preprocessing.
# sub-001 has NO Empatica data (dataset has E4 for 121/130 subjects
# only) — that's why your channels.tsv filter came back empty.
# Use a subject that actually has E4, e.g. sub-005.
# ============================================================

import os
import pandas as pd

DATASET_PATH = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0"

def check_subject(sub):
    tsv_path = os.path.join(DATASET_PATH, sub, "eeg", f"{sub}_task-sleep_channels.tsv")
    if not os.path.exists(tsv_path):
        print(f"{sub}: no channels.tsv found")
        return None
    df = pd.read_csv(tsv_path, sep='\t')
    emp_rows = df[df['name'].str.contains('Emp', case=False, na=False)]
    if emp_rows.empty:
        print(f"{sub}: no Empatica channels (this subject has no E4 recording)")
        return None
    print(f"\n{sub}: Empatica channels found —")
    print(emp_rows[['name', 'sampling_frequency', 'units', 'description']].to_string(index=False))
    return emp_rows

# Try a handful of subjects known/likely to have E4 (from your earlier scan)
candidates = ["sub-001", "sub-005", "sub-007", "sub-008", "sub-009"]

found = False
for sub in candidates:
    result = check_subject(sub)
    if result is not None:
        found = True

if not found:
    print("\nNone of the tried subjects had E4 — loop over all_subs and check every one:")
    print("""
all_subs = sorted([d for d in os.listdir(DATASET_PATH)
                    if d.startswith('sub-') and os.path.isdir(os.path.join(DATASET_PATH, d))])
for sub in all_subs:
    check_subject(sub)
""")

print("""
WHAT TO LOOK FOR:
  Known Empatica E4 device specs (manufacturer datasheet):
    BVP  : 64 Hz  (waveform)
    EDA  : 4 Hz
    TEMP : 4 Hz   (skin temperature, slow drift)
    HR   : 1 Hz   (derived, sliding 10s window over BVP/PPG)
    ACC  : 32 Hz
  If channels.tsv confirms HR=1Hz and TEMP=4Hz, then your current
  preprocessing (interpolating everything to 64Hz = 1920 samples/epoch)
  is storing ~30x and ~16x redundant/interpolated values for HR and TEMP
  respectively. That redundancy doesn't add information, it just gives
  the CNN more parameters to overfit noise/interpolation artifacts on.
""")

sub-001: no Empatica channels (this subject has no E4 recording)

sub-005: Empatica channels found —
    name  sampling_frequency    units                    description
Emp_ACCX                32.0     g/64 Device: Empatica E4 (Empatica)
Emp_ACCY                32.0     g/64 Device: Empatica E4 (Empatica)
Emp_ACCZ                32.0     g/64 Device: Empatica E4 (Empatica)
 Emp_EDA                 4.0       µS Device: Empatica E4 (Empatica)
 Emp_BVP                64.0 Unitless Device: Empatica E4 (Empatica)
Emp_TEMP                 4.0       °C Device: Empatica E4 (Empatica)
  Emp_HR                 1.0      bpm Device: Empatica E4 (Empatica)

sub-007: Empatica channels found —
    name  sampling_frequency    units                    description
Emp_ACCX                32.0     g/64 Device: Empatica E4 (Empatica)
Emp_ACCY                32.0     g/64 Device: Empatica E4 (Empatica)
Emp_ACCZ                32.0     g/64 Device: Empatica E4 (Empatica)
 Emp_EDA                 4.0       